In [3]:
import duckdb
import pandas as pd
from pathlib import Path

In [4]:
PROCESSED_DIR = Path("../data/processed")

In [5]:
con = duckdb.connect()

In [6]:
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

benchmark_clean.csv
ef_clean.csv
hd_clean.csv
saec_clean.csv
unitid_crosswalk.csv


In [7]:
con.execute("""
CREATE OR REPLACE TABLE benchmark AS
SELECT *
FROM read_csv_auto('../data/processed/benchmark_clean.csv')
""")

In [25]:
con.execute("""
CREATE OR REPLACE TABLE hd AS
SELECT *
FROM read_csv_auto('../data/processed/hd_clean.csv')
""")

In [33]:
con.execute("""
CREATE OR REPLACE TABLE ef AS
SELECT *
FROM read_csv_auto('../data/processed/ef_clean.csv')
""")

In [41]:
con.execute("""
CREATE OR REPLACE TABLE saec AS
SELECT *
FROM read_csv_auto('../data/processed/saec_clean.csv')
""")

In [49]:
con.execute("""
CREATE OR REPLACE TABLE unitid AS
SELECT *
FROM read_csv_auto('../data/processed/unitid_crosswalk.csv')
""")

In [58]:
con.sql("""
SHOW TABLES
""")

┌───────────┐
│   name    │
│  varchar  │
├───────────┤
│ benchmark │
│ ef        │
│ hd        │
│ saec      │
│ united    │
│ unitid    │
└───────────┘

In [62]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT UNITID) AS unique_universities
FROM benchmark
""").df()

,rows,unique_universities
0,32,32


In [63]:
for table in ["benchmark", "hd", "ef", "saec"]:

    result = con.sql(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT UNITID) AS unique_unitid
        FROM {table}
    """).df()

    print(table)
    print(result)
    print()
    

benchmark
   rows  unique_unitid
0    32             32

hd
   rows  unique_unitid
0  6072           6072

ef
   rows  unique_unitid
0  5842           5842

saec
   rows  unique_unitid
0  3927           3927



In [72]:
ucm_test = con.sql("""
SELECT
    b.uni_id,
    b.uni_name,
    b.UNITID,

    h.CITY,
    h.STABBR,
    h.CONTROL,
    h.INSTSIZE,
    h.LOCALE,
    h.CARNEGIEIC,
    h.CARNEGIESIZE,

    e.uni_total_enrollment,
    e.uni_total_nonresident_enrollment,
    e.uni_total_nonresident_pct,
    e.uni_undergrad_enrollment,
    e.uni_undergrad_nonresident_enrollment,
    e.uni_graduate_enrollment,
    e.uni_graduate_nonresident_enrollment,

    s.Pell_PCT,
    s.Access_Ratio,
    s."8Yr_MedEarnings",
    s.Earnings_Ratio,
    s.SAEC25

FROM benchmark AS b

LEFT JOIN hd AS h
    ON b.UNITID = h.UNITID

LEFT JOIN ef AS e
    ON b.UNITID = e.UNITID

LEFT JOIN saec AS s
    ON b.UNITID = s.UNITID

WHERE b.UNITID = 176965
""").df()